# NV-to-lab calibration from fixed left ODMR lines: response matrix and Schloss-style \(A\)

This notebook rewrites the previous geometry notebook around the quantity used by **Schloss et al.**: the **line-center response** of selected ODMR resonances, not the pair splittings.

## What this notebook computes

1. The **measured displacement-response matrix**
\[
R_{\mathrm{disp}}=\frac{\partial \boldsymbol{\nu}_{\mathrm{left}}}{\partial \mathbf r},
\]
where \(\boldsymbol{\nu}_{\mathrm{left}}\) contains the four **left-side** ODMR line centers and \(\mathbf r=(x,y,z)\) is the magnet displacement in the lab frame.

2. A **Schloss-style field-response matrix**
\[
A=\frac{\partial \boldsymbol{\nu}_{\mathrm{left}}}{\partial \mathbf B},
\]
evaluated at the chosen zero-offset operating point.

The first matrix is extracted **directly from your displacement scans**.  
The second matrix is obtained by **linearizing the NV spin Hamiltonian** around the fitted bias field and the fitted zero-offset line centers, following the logic of Eq. (1) in Schloss et al.

## Fixed dip-to-axis assignment used here

From your experimental knowledge of the [100]-cut sample and the spectra:

- outermost left dip \(\rightarrow d3\)
- second left dip \(\rightarrow d2\)
- third left dip \(\rightarrow d4\)
- left dip closest to the center \(\rightarrow d1\)

So the left-line order throughout the notebook is
\[
[d3,\ d2,\ d4,\ d1].
\]

## Theory used

For each NV orientation \(i\), we model the electron-spin Hamiltonian as
\[
H_i/h = (D+M_{z,i})S_z^2 + \gamma_e\,\mathbf B_i\cdot \mathbf S,
\]
where:

- \(D\) is the zero-field splitting,
- \(M_{z,i}\) is the fitted axial shift of orientation \(i\),
- \(\mathbf B_i\) is the lab magnetic field expressed in the local NV frame,
- \(\gamma_e \approx 28.02495164\,\mathrm{MHz/mT}\).

We then diagonalize the spin-1 Hamiltonian numerically and use finite differences to compute
\[
A_{ij}=\frac{\partial \nu_i}{\partial B_j}.
\]

This follows the **Hamiltonian linearization strategy of Schloss et al.**, where the resonance frequencies are linearized around the measured operating point to form the \(4\times3\) matrix \(A\).

### References
- J. M. Schloss *et al.*, **Simultaneous Broadband Vector Magnetometry Using Solid-State Spins**, *Phys. Rev. Applied* **10**, 034044 (2018).  
- J. F. Barry *et al.*, **Sensitivity optimization for NV-diamond magnetometry**, *Rev. Mod. Phys.* **92**, 015004 (2020).  
- D. Lönard *et al.*, **Limits of absolute vector magnetometry with NV centers in diamond** (2025/2026).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from pathlib import Path
from scipy.stats import linregress
from scipy.optimize import least_squares

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

GAMMA_E_MHZ_PER_MT = 28.02495164
LEFT_DIP_ORDER = ["d3", "d2", "d4", "d1"]  # outermost -> innermost on left side

OUTDIR = Path("./outputs")
OUTDIR.mkdir(exist_ok=True)


## Data source

The notebook can work in either of two ways:

1. **Preferred for this notebook:** read a single run-summary CSV if you already exported one.
2. Otherwise, reconstruct the same summary table directly from the existing folder structure:
   - `../fitting_odmr/batch_fit_outputs_lorentzian/Variation Along X/...`
   - `../fitting_odmr/batch_fit_outputs_lorentzian/Variation Along Y/...`
   - `../fitting_odmr/batch_fit_outputs_lorentzian/Variation Along Z/...`

The notebook keeps the same folder access logic as your current version.


In [5]:
ROOT = Path("../fitting_odmr/batch_fit_outputs_lorentzian")

VARIATION_TO_AXIS = {
    "Variation Along X": "x",
    "Variation Along Y": "y",
    "Variation Along Z": "z",
}

for variation in VARIATION_TO_AXIS:
    path = ROOT / variation
    print(f"{variation}: {'FOUND' if path.exists() else 'MISSING'} -> {path}")


NameError: name 'Path' is not defined

## Build the four left-side line traces

We now switch from pair splittings to the four selected **left** ODMR line centers.  
For every run, the pair-to-axis assignment is fixed by your experimental knowledge:
\[
\text{pair 1}\to d3,\quad
\text{pair 2}\to d2,\quad
\text{pair 3}\to d4,\quad
\text{pair 4}\to d1.
\]

This is the key change relative to the old notebook.

The frequency shifts are defined relative to the zero-displacement reference **within the same scan direction**:
\[
\Delta \nu_i(\Delta r_j)=\nu_i(\Delta r_j)-\nu_i(0).
\]


In [ ]:
PAIR_TO_NV = {1: "d3", 2: "d2", 3: "d4", 4: "d1"}

left_rows = []
for _, row in summary_df.iterrows():
    for pair_id in [1, 2, 3, 4]:
        left_rows.append({
            "variation": row["variation"],
            "lab_axis": row["lab_axis"],
            "run_name": row["run_name"],
            "offset_mm": float(row["offset_mm"]),
            "pair_id": pair_id,
            "nv_axis": PAIR_TO_NV[pair_id],
            "nu_left_GHz": float(row[f"pair_{pair_id}_left_x1_GHz"]),
            "nu_right_GHz": float(row[f"pair_{pair_id}_right_x1_GHz"]),
            "splitting_MHz": float(row[f"pair_{pair_id}_splitting_MHz"]),
            "pair_center_minus_D_eff_MHz": float(row[f"pair_{pair_id}_center_minus_D_eff_MHz"]),
            "D_eff_GHz": float(row["D_eff_GHz"]),
        })

left_lines_long = pd.DataFrame(left_rows)
left_lines_long["nu_left_MHz"] = 1e3 * left_lines_long["nu_left_GHz"]
left_lines_long["nu_right_MHz"] = 1e3 * left_lines_long["nu_right_GHz"]

zero_ref = (
    left_lines_long[np.isclose(left_lines_long["offset_mm"], 0.0)]
    .groupby(["variation", "lab_axis", "nv_axis"], as_index=False)["nu_left_MHz"]
    .mean()
    .rename(columns={"nu_left_MHz": "nu0_left_MHz"})
)

left_lines_long = left_lines_long.merge(zero_ref, on=["variation", "lab_axis", "nv_axis"], how="left")
left_lines_long["delta_nu_left_MHz"] = left_lines_long["nu_left_MHz"] - left_lines_long["nu0_left_MHz"]

display(left_lines_long.head(12))


## Visual inspection: left-line shifts vs displacement

This replaces the old splitting plot.  
We keep only the four left-side line centers, in the order:
\[
d3,\ d2,\ d4,\ d1.
\]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax_plot, (variation, lab_axis) in zip(axes, VARIATION_TO_AXIS.items()):
    sub = left_lines_long[left_lines_long["variation"] == variation]

    for nv_axis in LEFT_DIP_ORDER:
        d = sub[sub["nv_axis"] == nv_axis].sort_values("offset_mm")
        ax_plot.plot(d["offset_mm"], d["delta_nu_left_MHz"], marker="o", label=nv_axis)

    ax_plot.set_title(variation)
    ax_plot.set_xlabel("Displacement (mm)")
    ax_plot.set_ylabel(r"$\Delta \nu_{\mathrm{left}}$ (MHz)")
    ax_plot.legend(title="NV axis")

plt.tight_layout()
plt.savefig(OUTDIR / "left_dip_shifts_vs_displacement.png", bbox_inches="tight")
plt.show()


## Linear fits and measured response matrix

For each left ODMR line \(i\) and each scan axis \(j\), we fit
\[
\Delta \nu_i(\Delta r_j)\approx b_{ij}+R_{ij}\,\Delta r_j,
\]
so that
\[
R_{ij}=\frac{\partial \nu_i}{\partial r_j}.
\]

This gives the measured \(4\times3\) response matrix \(R_{\mathrm{disp}}\) in units of MHz/mm.


In [ ]:
def fit_line_slope(df_axis_line: pd.DataFrame, max_abs_offset_mm=None) -> dict:
    d = df_axis_line.sort_values("offset_mm").copy()

    if max_abs_offset_mm is not None:
        d = d[np.abs(d["offset_mm"]) <= max_abs_offset_mm].copy()

    x = d["offset_mm"].to_numpy(dtype=float)
    y = d["delta_nu_left_MHz"].to_numpy(dtype=float)

    if len(d) < 2:
        raise ValueError("Need at least two points for a linear fit.")

    fit = linregress(x, y)

    return {
        "n_points": len(d),
        "offset_min_mm": float(np.min(x)),
        "offset_max_mm": float(np.max(x)),
        "slope_MHz_per_mm": float(fit.slope),
        "intercept_MHz": float(fit.intercept),
        "r_value": float(fit.rvalue),
        "r_squared": float(fit.rvalue**2),
        "p_value": float(fit.pvalue),
        "slope_stderr": float(fit.stderr),
        "intercept_stderr": float(getattr(fit, "intercept_stderr", np.nan)),
    }

slope_rows = []
for variation, lab_axis in VARIATION_TO_AXIS.items():
    sub_var = left_lines_long[left_lines_long["variation"] == variation]
    for nv_axis in LEFT_DIP_ORDER:
        sub_line = sub_var[sub_var["nv_axis"] == nv_axis]
        fit_res = fit_line_slope(sub_line, max_abs_offset_mm=None)
        slope_rows.append({
            "variation": variation,
            "lab_axis": lab_axis,
            "nv_axis": nv_axis,
            **fit_res,
        })

slopes_df = pd.DataFrame(slope_rows).sort_values(["nv_axis", "lab_axis"]).reset_index(drop=True)
display(slopes_df)

R_disp = (
    slopes_df.pivot(index="nv_axis", columns="lab_axis", values="slope_MHz_per_mm")
    .reindex(index=LEFT_DIP_ORDER, columns=["x", "y", "z"])
)

R_disp_err = (
    slopes_df.pivot(index="nv_axis", columns="lab_axis", values="slope_stderr")
    .reindex(index=LEFT_DIP_ORDER, columns=["x", "y", "z"])
)

print("Measured displacement-response matrix R_disp [MHz/mm]:")
display(R_disp)

print("Slope standard errors [MHz/mm]:")
display(R_disp_err)


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.2))
im = ax.imshow(R_disp.to_numpy(dtype=float), aspect="auto")
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["x", "y", "z"])
ax.set_yticks(range(4))
ax.set_yticklabels(LEFT_DIP_ORDER)
ax.set_title(r"$R_{\mathrm{disp}} = \partial \nu / \partial r$")

for i in range(R_disp.shape[0]):
    for j in range(R_disp.shape[1]):
        val = R_disp.iloc[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center")

fig.colorbar(im, ax=ax, label="MHz/mm")
plt.tight_layout()
plt.savefig(OUTDIR / "R_disp_heatmap.png", bbox_inches="tight")
plt.show()


## Reference operating point for the Hamiltonian linearization

To compute the Schloss-style matrix
\[
A=\frac{\partial \boldsymbol{\nu}}{\partial \mathbf B},
\]
we need a single zero-offset operating point.

This notebook chooses the best available zero-offset run in the following order:

1. prefer `Variation Along X`,
2. then `Variation Along Y`,
3. then `Variation Along Z`,

while favoring small fit residuals and rejecting obvious outliers.

This is deliberate because your pasted summary clearly shows that the `Variation Along Z` zero run is an outlier in \(D_\mathrm{eff}\) and linewidth.


In [ ]:
def choose_reference_run(summary_df: pd.DataFrame) -> pd.Series:
    z = summary_df[np.isclose(summary_df["offset_mm"], 0.0)].copy()

    # fill missing quality columns if absent
    for col in ["global_rms", "lorentz_fwhm_MHz"]:
        if col not in z.columns:
            z[col] = np.nan

    # simple preference ordering
    pref = {"Variation Along X": 0, "Variation Along Y": 1, "Variation Along Z": 2}
    z["variation_pref"] = z["variation"].map(pref).fillna(99)

    # robust D filter if multiple zero runs exist
    D_med = z["D_eff_GHz"].median()
    z["D_dev"] = np.abs(z["D_eff_GHz"] - D_med)

    z = z.sort_values(["variation_pref", "D_dev", "global_rms", "lorentz_fwhm_MHz"]).reset_index(drop=True)
    return z.iloc[0]

ref_row = choose_reference_run(summary_df)
display(ref_row.to_frame().T)

print("Chosen reference run:")
print(f"variation = {ref_row['variation']}")
print(f"run_name   = {ref_row['run_name']}")
print(f"offset_mm  = {ref_row['offset_mm']}")


## [100]-cut geometry and fixed NV-axis set

For a [100]-cut diamond with lab \(z\) along the surface normal / optical axis, a convenient tetrahedral NV-axis template is
\[
\begin{aligned}
d3 &= \left(+\sqrt{\tfrac23},\,0,\,+\tfrac{1}{\sqrt3}\right),\\
d4 &= \left(-\sqrt{\tfrac23},\,0,\,+\tfrac{1}{\sqrt3}\right),\\
d2 &= \left(0,\,+\sqrt{\tfrac23},\,-\tfrac{1}{\sqrt3}\right),\\
d1 &= \left(0,\,-\sqrt{\tfrac23},\,-\tfrac{1}{\sqrt3}\right).
\end{aligned}
\]

This enforces the geometry you specified: two orientations above the plane \((d3,d4)\) and two below \((d1,d2)\).


In [ ]:
a = np.sqrt(2/3)
b = 1/np.sqrt(3)

NV_AXES_LAB = {
    "d3": np.array([+a,  0.0, +b]),
    "d2": np.array([ 0.0, +a, -b]),
    "d4": np.array([-a,  0.0, +b]),
    "d1": np.array([ 0.0, -a, -b]),
}

N_lab = np.vstack([NV_AXES_LAB[k] for k in LEFT_DIP_ORDER])

display(pd.DataFrame(N_lab, index=LEFT_DIP_ORDER, columns=["lab_x", "lab_y", "lab_z"]))


## Estimate the bias field from the reference spectrum

At the chosen zero-offset operating point, the four splittings give the four projected-field magnitudes
\[
|n_i\cdot B_0| \approx \frac{\Delta \nu_i}{2\gamma_e}.
\]

Because the sign of each projection is not directly visible from a single static ODMR spectrum, we test all sign patterns and choose the one with the smallest least-squares residual.  
That gives an initial \(B_0\), which we then refine by fitting the full spin-1 Hamiltonian to the eight measured line centers of the reference spectrum.


In [ ]:
ref_pairs = []
for pair_id in [1, 2, 3, 4]:
    nv_axis = PAIR_TO_NV[pair_id]
    ref_pairs.append({
        "pair_id": pair_id,
        "nv_axis": nv_axis,
        "nu_left_GHz": float(ref_row[f"pair_{pair_id}_left_x1_GHz"]),
        "nu_right_GHz": float(ref_row[f"pair_{pair_id}_right_x1_GHz"]),
        "splitting_MHz": float(ref_row[f"pair_{pair_id}_splitting_MHz"]),
        "Mz_MHz": float(ref_row[f"pair_{pair_id}_center_minus_D_eff_MHz"]),
    })
ref_pairs_df = pd.DataFrame(ref_pairs).set_index("nv_axis").loc[LEFT_DIP_ORDER].reset_index()
display(ref_pairs_df)

proj_abs_mT = ref_pairs_df["splitting_MHz"].to_numpy(dtype=float) / (2 * GAMMA_E_MHZ_PER_MT)

candidate_sign_rows = []
for bits in range(16):
    signs = np.array([1 if (bits >> k) & 1 else -1 for k in range(4)], dtype=float)
    target_proj = signs * proj_abs_mT
    B_fit, *_ = np.linalg.lstsq(N_lab, target_proj, rcond=None)
    resid = N_lab @ B_fit - target_proj
    candidate_sign_rows.append({
        "signs": tuple(int(s) for s in signs),
        "Bx0_mT": B_fit[0],
        "By0_mT": B_fit[1],
        "Bz0_mT": B_fit[2],
        "proj_resid_rms_mT": float(np.sqrt(np.mean(resid**2))),
    })

sign_scan_df = pd.DataFrame(candidate_sign_rows).sort_values("proj_resid_rms_mT").reset_index(drop=True)
display(sign_scan_df.head(10))

best_signs = np.array(sign_scan_df.iloc[0]["signs"], dtype=float)
B0_guess_mT = sign_scan_df.loc[0, ["Bx0_mT", "By0_mT", "Bz0_mT"]].to_numpy(dtype=float)

print("Best low-field projection signs for LEFT lines:", best_signs)
print("Initial B0 guess [mT]:", B0_guess_mT)


## Numerical Hamiltonian fit and numerical \(A\)

We now refine the bias field by fitting the full spin-1 Hamiltonian to the **eight** measured line centers of the reference spectrum.

For each NV orientation \(i\), we use
\[
H_i/h = (D+M_{z,i})S_z^2 + \gamma_e\,\mathbf B_i\cdot \mathbf S,
\]
with \(D\) from the chosen reference run and \(M_{z,i}\) from the fitted pair centers:
\[
M_{z,i} = \nu_{c,i} - D.
\]

The lower transition is the measured **left** line and the upper transition is the measured **right** line.  
Finally, we compute
\[
A_{ij}=\frac{\partial \nu_i}{\partial B_j}
\]
numerically by finite differences around the fitted \(B_0\).


In [ ]:
# Spin-1 operators in the {|+1>, |0>, |-1>} basis
Sx = (1 / np.sqrt(2)) * np.array([[0, 1, 0],
                                  [1, 0, 1],
                                  [0, 1, 0]], dtype=complex)

Sy = (1 / np.sqrt(2)) * np.array([[0, -1j, 0],
                                  [1j,  0, -1j],
                                  [0,  1j, 0]], dtype=complex)

Sz = np.array([[1, 0, 0],
               [0, 0, 0],
               [0, 0,-1]], dtype=complex)

Sz2 = Sz @ Sz

def make_local_frame(nv_axis_lab: np.ndarray):
    ez = nv_axis_lab / np.linalg.norm(nv_axis_lab)
    tmp = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(tmp, ez)) > 0.9:
        tmp = np.array([1.0, 0.0, 0.0])
    ex = np.cross(tmp, ez)
    ex = ex / np.linalg.norm(ex)
    ey = np.cross(ez, ex)
    return ex, ey, ez

LOCAL_FRAMES = {k: make_local_frame(NV_AXES_LAB[k]) for k in LEFT_DIP_ORDER}

def nv_transition_freqs_GHz(B_lab_mT: np.ndarray, D_GHz: float, Mz_MHz: float, nv_axis: str):
    ex, ey, ez = LOCAL_FRAMES[nv_axis]
    B_local_mT = np.array([
        np.dot(B_lab_mT, ex),
        np.dot(B_lab_mT, ey),
        np.dot(B_lab_mT, ez),
    ])

    H_MHz = (D_GHz * 1e3 + Mz_MHz) * Sz2 + GAMMA_E_MHZ_PER_MT * (
        B_local_mT[0] * Sx + B_local_mT[1] * Sy + B_local_mT[2] * Sz
    )

    evals = np.linalg.eigvalsh(H_MHz)
    evals = np.sort(evals.real)

    nu_left_GHz = (evals[1] - evals[0]) / 1e3
    nu_right_GHz = (evals[2] - evals[0]) / 1e3
    return np.array([nu_left_GHz, nu_right_GHz], dtype=float)

D_ref_GHz = float(ref_row["D_eff_GHz"])
Mz_ref_MHz = ref_pairs_df.set_index("nv_axis")["Mz_MHz"].to_dict()

def residuals_B_only(B_lab_mT):
    rr = []
    for nv_axis in LEFT_DIP_ORDER:
        pred = nv_transition_freqs_GHz(B_lab_mT, D_ref_GHz, Mz_ref_MHz[nv_axis], nv_axis)
        meas = ref_pairs_df.loc[ref_pairs_df["nv_axis"] == nv_axis, ["nu_left_GHz", "nu_right_GHz"]].iloc[0].to_numpy(dtype=float)
        rr.extend(pred - meas)
    return np.array(rr, dtype=float)

lsq = least_squares(residuals_B_only, x0=B0_guess_mT, method="trf")
B0_fit_mT = lsq.x

print("Reference D [GHz]:", D_ref_GHz)
print("Fitted B0 [mT]:", B0_fit_mT)
print("Reference-fit residual RMS [MHz]:", 1e3 * np.sqrt(np.mean(lsq.fun**2)))

fit_compare_rows = []
for nv_axis in LEFT_DIP_ORDER:
    pred = nv_transition_freqs_GHz(B0_fit_mT, D_ref_GHz, Mz_ref_MHz[nv_axis], nv_axis)
    meas = ref_pairs_df.loc[ref_pairs_df["nv_axis"] == nv_axis, ["nu_left_GHz", "nu_right_GHz"]].iloc[0].to_numpy(dtype=float)
    fit_compare_rows.append({
        "nv_axis": nv_axis,
        "meas_left_GHz": meas[0],
        "pred_left_GHz": pred[0],
        "meas_right_GHz": meas[1],
        "pred_right_GHz": pred[1],
        "left_resid_MHz": 1e3 * (pred[0] - meas[0]),
        "right_resid_MHz": 1e3 * (pred[1] - meas[1]),
        "n_dot_B_mT": float(np.dot(NV_AXES_LAB[nv_axis], B0_fit_mT)),
    })

fit_compare_df = pd.DataFrame(fit_compare_rows)
display(fit_compare_df)


In [ ]:
# Low-field A from the fitted sign pattern of n_i · B0 for the LEFT branch:
proj_signs = np.sign(fit_compare_df["n_dot_B_mT"].to_numpy(dtype=float))
proj_signs[proj_signs == 0] = 1.0

A_lowfield = (-proj_signs[:, None]) * GAMMA_E_MHZ_PER_MT * N_lab
A_lowfield_df = pd.DataFrame(A_lowfield, index=LEFT_DIP_ORDER, columns=["Bx", "By", "Bz"])

print("Low-field Schloss-style A matrix [MHz/mT]:")
display(A_lowfield_df)

def left_transition_GHz(B_lab_mT: np.ndarray, nv_axis: str) -> float:
    return nv_transition_freqs_GHz(B_lab_mT, D_ref_GHz, Mz_ref_MHz[nv_axis], nv_axis)[0]

def numerical_A_matrix(B_lab_mT: np.ndarray, step_mT: float = 1e-4):
    A = np.zeros((4, 3), dtype=float)
    for i, nv_axis in enumerate(LEFT_DIP_ORDER):
        for j in range(3):
            dB = np.zeros(3, dtype=float)
            dB[j] = step_mT
            fp = left_transition_GHz(B_lab_mT + dB, nv_axis)
            fm = left_transition_GHz(B_lab_mT - dB, nv_axis)
            A[i, j] = (fp - fm) * 1e3 / (2 * step_mT)  # MHz / mT
    return A

A_num = numerical_A_matrix(B0_fit_mT, step_mT=1e-4)
A_num_df = pd.DataFrame(A_num, index=LEFT_DIP_ORDER, columns=["Bx", "By", "Bz"])

print("Numerically linearized Schloss-style A matrix [MHz/mT]:")
display(A_num_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

for ax, mat, title in zip(
    axes,
    [A_lowfield_df.to_numpy(dtype=float), A_num_df.to_numpy(dtype=float)],
    [r"Low-field $A$", r"Numerically linearized $A$"]
):
    im = ax.imshow(mat, aspect="auto")
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["Bx", "By", "Bz"])
    ax.set_yticks(range(4))
    ax.set_yticklabels(LEFT_DIP_ORDER)
    ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center")
    fig.colorbar(im, ax=ax, shrink=0.85, label="MHz/mT")

plt.tight_layout()
plt.savefig(OUTDIR / "A_matrix_comparison.png", bbox_inches="tight")
plt.show()


## Relation between the measured matrix and the field-response matrix

The measured displacement-response matrix and the Hamiltonian matrix are related by
\[
R_{\mathrm{disp}} = A\,G,
\qquad
G=\frac{\partial \mathbf B}{\partial \mathbf r}.
\]

So:
- \(R_{\mathrm{disp}}\) comes directly from your displacement scans,
- \(A\) comes from the NV Hamiltonian at the operating point,
- \(G\) is the local field gradient produced by moving the magnet.

If later you want the local field-gradient matrix itself, you can compute
\[
G \approx A^+ R_{\mathrm{disp}},
\]
using the left Moore-Penrose pseudoinverse \(A^+\), exactly in the same spirit as Schloss et al.


In [ ]:
A_num_plus = np.linalg.pinv(A_num_df.to_numpy(dtype=float))
A_num_plus_df = pd.DataFrame(A_num_plus, index=["Bx", "By", "Bz"], columns=LEFT_DIP_ORDER)

G_est = A_num_plus @ R_disp.to_numpy(dtype=float)
G_est_df = pd.DataFrame(G_est, index=["Bx", "By", "Bz"], columns=["dx", "dy", "dz"])

print("Pseudoinverse A^+ [mT/MHz]:")
display(A_num_plus_df)

print("Estimated local field-gradient matrix G = dB/dr [mT/mm]:")
display(G_est_df)


In [ ]:
R_disp.to_csv(OUTDIR / "R_disp_MHz_per_mm.csv")
R_disp_err.to_csv(OUTDIR / "R_disp_errors_MHz_per_mm.csv")
A_lowfield_df.to_csv(OUTDIR / "A_lowfield_MHz_per_mT.csv")
A_num_df.to_csv(OUTDIR / "A_numerical_MHz_per_mT.csv")
A_num_plus_df.to_csv(OUTDIR / "A_pseudoinverse_mT_per_MHz.csv")
G_est_df.to_csv(OUTDIR / "G_est_mT_per_mm.csv")
left_lines_long.to_csv(OUTDIR / "left_lines_long.csv", index=False)
slopes_df.to_csv(OUTDIR / "left_line_slopes.csv", index=False)
fit_compare_df.to_csv(OUTDIR / "reference_spectrum_fit_compare.csv", index=False)

print("Saved outputs to:", OUTDIR.resolve())
print("\nR_disp [MHz/mm]:")
print(R_disp.to_string(float_format=lambda x: f"{x: .6f}"))
print("\nA_num [MHz/mT]:")
print(A_num_df.to_string(float_format=lambda x: f"{x: .6f}"))
print("\nG_est [mT/mm]:")
print(G_est_df.to_string(float_format=lambda x: f"{x: .6f}"))
